# Train XGBoost direction_5d

Notebook train XGBoost cho bai toan du bao xu huong 5 ngay, tach theo time series va luu model/features/feature importance.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, log_loss
from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8-darkgrid')

if 'df' not in globals():
    df = pd.read_csv('all_stocks_train.csv', parse_dates=['trading_date'])

df = df.sort_values(['trading_date', 'symbol']).reset_index(drop=True)
df = df.dropna(subset=['direction_5d']).copy()

In [ ]:
technical_features = [
    'sma_10', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist',
    'adx_14', 'di_plus', 'di_minus', 'rsi_14', 'stoch_k', 'stoch_d', 'williams_r',
    'bb_upper', 'bb_mid', 'bb_lower', 'bb_pct_b', 'bb_width', 'atr_14', 'obv', 'vwap',
    'volume_sma_20', 'volume_ratio', 'candle_body_size', 'candle_upper_wick',
    'candle_lower_wick', 'is_doji', 'is_hammer', 'is_bull_engulfing', 'is_bear_engulfing',
    'is_morning_star', 'is_evening_star'
]

split_date = pd.Timestamp('2025-10-01')
train_df = df[df['trading_date'] < split_date].copy()
test_df = df[df['trading_date'] >= split_date].copy()

train_df = train_df.dropna(subset=technical_features)
test_df = test_df.dropna(subset=technical_features)

val_cutoff = train_df['trading_date'].quantile(0.85)
train_fit = train_df[train_df['trading_date'] < val_cutoff].copy()
val_df = train_df[train_df['trading_date'] >= val_cutoff].copy()

X_train = train_fit[technical_features]
y_train = train_fit['direction_5d'].astype(int)
X_val = val_df[technical_features]
y_val = val_df['direction_5d'].astype(int)
X_test = test_df[technical_features]
y_test = test_df['direction_5d'].astype(int)

def summarize(name, frame):
    dist = frame['direction_5d'].value_counts(normalize=True).sort_index().mul(100).round(2).to_dict()
    return {
        'name': name,
        'rows': len(frame),
        'from': frame['trading_date'].min(),
        'to': frame['trading_date'].max(),
        'direction_0_pct': dist.get(0.0, dist.get(0, 0.0)),
        'direction_1_pct': dist.get(1.0, dist.get(1, 0.0)),
    }

pd.DataFrame([
    summarize('train_fit', train_fit),
    summarize('val', val_df),
    summarize('test', test_df),
])

In [ ]:
model = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.2,
    objective='binary:logistic',
    eval_metric=['logloss', 'auc'],
    random_state=42,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    early_stopping_rounds=50,
    verbose=50,
)

evals = model.evals_result()

In [ ]:
best_iteration = getattr(model, 'best_iteration', None)
print('best_iteration =', best_iteration)

train_loss = evals['validation_0']['logloss']
val_loss = evals['validation_1']['logloss']

plt.figure(figsize=(10, 5))
plt.plot(train_loss, label='train logloss')
plt.plot(val_loss, label='val logloss')
plt.title('Learning curve')
plt.xlabel('Boosting round')
plt.ylabel('Log loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print('Accuracy:', round(accuracy_score(y_test, test_pred), 4))
print('ROC-AUC :', round(roc_auc_score(y_test, test_proba), 4))
print('LogLoss :', round(log_loss(y_test, test_proba), 4))
print('\nClassification report:\n')
print(classification_report(y_test, test_pred, digits=4))

importance = pd.DataFrame({
    'feature': technical_features,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
display(importance.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 feature importance')
plt.tight_layout()
plt.show()

In [ ]:
MODEL_OUTPUT_DIR = Path('/content/drive/MyDrive/stock_ai')
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(MODEL_OUTPUT_DIR / 'xgb_direction_5d.pkl', 'wb') as fh:
    pickle.dump(model, fh)

with open(MODEL_OUTPUT_DIR / 'features.json', 'w', encoding='utf-8') as fh:
    json.dump(technical_features, fh, ensure_ascii=False, indent=2)

importance.to_csv(MODEL_OUTPUT_DIR / 'feature_importance.csv', index=False, encoding='utf-8-sig')
print('Da luu model, features, feature importance vao Drive:', MODEL_OUTPUT_DIR)

In [ ]:
quick_test = test_df[['symbol', 'trading_date', 'direction_5d']].copy()
quick_test['ai_score'] = test_proba
sample_symbols = quick_test['symbol'].drop_duplicates().sample(min(5, quick_test['symbol'].nunique()), random_state=42)
quick_view = quick_test[quick_test['symbol'].isin(sample_symbols)].sort_values(['symbol', 'trading_date'])
quick_view['thuc_te'] = quick_view['direction_5d'].map({1: 'Tang', 0: 'Giam'})
display(quick_view[['symbol', 'trading_date', 'ai_score', 'thuc_te']].tail(100))

print('Nhan xet nhanh:')
print('- Neu score > 0.65 tap trung vao cac ngay thuc_te tang nhieu hon thi model hop ly.')
print('- Neu score dao dong sat 0.5 va nham lien tuc theo cung 1 nhom ma/nganh thi can bo sung feature hoac tune threshold.')